# NetCDF & Shapefile TPU Training Pipeline
This notebook is optimized to run on Google Colab TPUs. It loads the `.nc` file, clips it to the Indore shapefile, and trains a ConvLSTM spatial model.

In [ ]:
!pip install xarray netCDF4 geopandas rioxarray tensorflow

### Initialize TPU & Mount Drive

In [ ]:
import tensorflow as tf
import os

# Initialize TPU
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver() 
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print("TPU initialized successfully!")
except ValueError:
    print("No TPU found. Falling back to CPU/GPU.")
    strategy = tf.distribute.get_strategy()

# Mount Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    data_dir = '/content/drive/MyDrive/RainCast_Data/'
except ImportError:
    data_dir = './Data/'

### Load NetCDF & Setup Spatial Dimensions

In [ ]:
import xarray as xr
import geopandas as gpd
import rioxarray

nc_file = os.path.join(data_dir, 'indore_era5.nc')

# Open the NetCDF file downloaded in the previous notebook
ds = xr.open_dataset(nc_file)

# Tell rioxarray what dimensions represent X (lon) and Y (lat) so we can clip it
ds = ds.rio.write_crs("epsg:4326")
ds.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
print(ds)

### Masking with Shapefile
Upload your `indore.shp` (and associated .shx, .dbf) to your drive data folder.

In [ ]:
# Uncomment this section when you have the shapefile uploaded!

# shp_path = os.path.join(data_dir, 'indore.shp')
# indore_shp = gpd.read_file(shp_path)

# Clip the grid to the exact shapefile boundaries. Pixels outside become NaN.
# clipped_ds = ds.rio.clip(indore_shp.geometry.values, indore_shp.crs, drop=True)

clipped_ds = ds  # Using unclipped grid for now
print("Data clipped and ready!")

### Data Preparation (Time Series Tensors)

In [ ]:
import numpy as np

# Extract variables to numpy arrays. Shape: (Time, Lat, Lon)
precip = clipped_ds['total_precipitation'].values
temp = clipped_ds['maximum_2m_air_temperature'].values

# Stack them into channels. Shape: (Time, Lat, Lon, Channels)
X_data = np.stack([precip, temp], axis=-1)

# Replace NaNs (outside the shapefile) with 0
X_data = np.nan_to_num(X_data)
print("Input Tensor shape:", X_data.shape)

### Build TPU Model (Spatio-Temporal U-Net)
Since we are training on a 3D grid over time, we use a U-Net architecture which is state-of-the-art for spatial weather mapping. It encodes spatial features and decodes them into precise rainfall maps.

In [ ]:
# Build model inside the TPU strategy scope
with strategy.scope():
    inputs = tf.keras.layers.Input(shape=(None, X_data.shape[1], X_data.shape[2], 2))
    
    # TimeDistributed applies the Conv2D to each time step independently
    # Encoder
    c1 = tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'))(inputs)
    c1 = tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'))(c1)
    
    # Bottleneck
    c2 = tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'))(c1)
    
    # Decoder (Skip connection from Encoder)
    u1 = tf.keras.layers.Concatenate()([c2, c1])
    c3 = tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'))(u1)
    c3 = tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'))(c3)
    
    # Temporal Smoothing (LSTM)
    # We collapse the spatial dimensions, run an LSTM to capture temporal dynamics, and reshape back
    shape = tf.keras.backend.int_shape(c3)
    flat = tf.keras.layers.Reshape((-1, shape[2] * shape[3] * shape[4]))(c3)
    lstm = tf.keras.layers.LSTM(shape[2] * shape[3] * shape[4], return_sequences=True)(flat)
    reshaped_lstm = tf.keras.layers.Reshape((-1, shape[2], shape[3], shape[4]))(lstm)
    
    # Output layer: Predicts 1 value (rainfall) per pixel in the grid
    outputs = tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(1, (1, 1), activation='relu', padding='same'))(reshaped_lstm)
    
    model = tf.keras.Model(inputs=[inputs], outputs=[outputs])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

model.summary()